In [1]:
!pip install openai datasets -q

In [2]:
import os
import json
import time
import re
import hashlib
import pandas as pd
from collections import defaultdict
from openai import OpenAI
from datasets import load_dataset

OPENAI_API_KEY = "OPENAI_API_KEY"

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-3.5-turbo"

print("Testing OpenAI connection...")
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "say hello"}]
)
print("OpenAI connected successfully")
print(test.choices[0].message.content)

print("\nLoading SWE-bench Lite dataset...")
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
print(f"Dataset loaded: {len(dataset)} tasks available")

Testing OpenAI connection...
OpenAI connected successfully
Hello! How can I assist you today?

Loading SWE-bench Lite dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/120k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/23 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset loaded: 300 tasks available


In [3]:
def call_openai_with_retry(messages, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.2
            )
            return response
        except Exception as e:
            error_msg = str(e)
            if "rate_limit" in error_msg or "429" in error_msg:
                print(f"  Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
                continue
            else:
                raise e
    raise Exception("Max retries exceeded")

print("Retry logic defined successfully")

Retry logic defined successfully


In [4]:
class EntityMemory:
    def __init__(self):
        self.records = []
        self.entity_graph = defaultdict(set)

    def extract_entities(self, text):
        entities = set()
        files = re.findall(r'[\w/]+\.py', text)
        entities.update(files)
        functions = re.findall(r'def\s+(\w+)', text)
        entities.update(functions)
        classes = re.findall(r'class\s+(\w+)', text)
        entities.update(classes)
        caps = re.findall(r'\b[A-Z][a-zA-Z]+\b', text)
        entities.update(caps)
        return entities

    def add_record(self, content, source, phase):
        entities = self.extract_entities(content)
        record = {
            "id": hashlib.sha256(content.encode()).hexdigest()[:8],
            "content": content,
            "source": source,
            "phase": phase,
            "entities": entities,
            "importance": len(entities)
        }
        self.records.append(record)
        for entity in entities:
            self.entity_graph[entity].add(record["id"])
        return record

    def retrieve(self, query, top_k=3):
        if not self.records:
            return []
        query_entities = self.extract_entities(query)
        scored = []
        for record in self.records:
            record_entities = record["entities"]
            if query_entities or record_entities:
                intersection = len(query_entities & record_entities)
                union = len(query_entities | record_entities)
                jaccard = intersection / union if union > 0 else 0
            else:
                jaccard = 0
            importance = record["importance"] / 10.0
            score = 0.7 * jaccard + 0.3 * importance
            scored.append((score, record))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [r for _, r in scored[:top_k]]

print("Entity Memory defined successfully")

Entity Memory defined successfully


In [5]:
PHASES = {
    "diagnosis": {
        "allowed_tools": ["read_file", "grep", "search"],
        "forbidden_tools": ["edit_file", "apply_patch"],
        "prompt": """You are in the DIAGNOSIS phase.
Your ONLY goal is to understand the issue and identify the root cause.
You can ONLY use these tools: read_file, grep, search.
You CANNOT edit any files in this phase.
When you have identified the root cause, set is_complete to true."""
    },
    "planning": {
        "allowed_tools": ["search", "read_file"],
        "forbidden_tools": ["edit_file", "apply_patch"],
        "prompt": """You are in the PLANNING phase.
Your ONLY goal is to propose a fix strategy based on your diagnosis.
You can ONLY use these tools: search, read_file.
You CANNOT edit any files in this phase.
When you have a clear plan, set is_complete to true."""
    },
    "patching": {
        "allowed_tools": ["edit_file", "apply_patch", "read_file"],
        "forbidden_tools": [],
        "prompt": """You are in the PATCHING phase.
Your ONLY goal is to apply the fix you planned.
You can use these tools: edit_file, apply_patch, read_file.
When you have applied the fix, set is_complete to true."""
    },
    "verification": {
        "allowed_tools": ["run_tests", "read_file"],
        "forbidden_tools": ["edit_file", "apply_patch"],
        "prompt": """You are in the VERIFICATION phase.
Your ONLY goal is to verify the fix works and no regressions exist.
You can ONLY use these tools: run_tests, read_file.
You CANNOT edit any files in this phase.
When verification is complete, set is_complete to true."""
    }
}

PHASE_ORDER = ["diagnosis", "planning", "patching", "verification"]
ALL_TOOLS = ["read_file", "grep", "search",
             "edit_file", "apply_patch", "run_tests"]

SYSTEM_PROMPT = """You are a software engineer fixing a GitHub issue.
You must follow the phase instructions exactly.
For each step respond in this exact JSON format:
{
    "thought": "your reasoning here",
    "tool": "tool_name",
    "args": {"arg1": "value1"},
    "is_complete": false
}
Set is_complete to true when you have finished the current phase.
Only respond with the JSON, nothing else."""


def run_baseline_agent(issue, repo):
    metrics = {
        "task_id": repo,
        "success": False,
        "total_steps": 0,
        "invalid_tool_calls": 0,
        "phase_violations": 0,
        "verification_executed": False,
        "total_tokens": 0,
        "avg_tool_reward": 0.0,
        "trajectory": []
    }

    system_prompt = """You are a software engineer fixing a GitHub issue.
You have access to these tools: read_file, grep, search,
edit_file, apply_patch, run_tests.
You can use any tool at any time in any order.

For each step respond in this exact JSON format:
{
    "thought": "your reasoning here",
    "tool": "tool_name",
    "args": {"arg1": "value1"},
    "is_complete": false
}
Set is_complete to true when you have finished.
Only respond with the JSON, nothing else."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Fix this issue in {repo}:\n\n{issue}"}
    ]

    for step in range(10):
        response = call_openai_with_retry(messages)
        raw = response.choices[0].message.content
        metrics["total_tokens"] += response.usage.total_tokens

        try:
            clean = raw.replace("```json","").replace("```","").strip()
            action = json.loads(clean)
        except:
            metrics["invalid_tool_calls"] += 1
            messages.append({"role": "assistant", "content": raw})
            messages.append({
                "role": "user",
                "content": "Your response was not valid JSON. Please respond with valid JSON only."
            })
            continue

        tool_used = action.get("tool", "")
        if tool_used not in ALL_TOOLS:
            metrics["invalid_tool_calls"] += 1
        if tool_used == "run_tests":
            metrics["verification_executed"] = True

        metrics["trajectory"].append({
            "step": metrics["total_steps"] + 1,
            "thought": action.get("thought", ""),
            "tool": tool_used,
            "args": action.get("args", {})
        })
        metrics["total_steps"] += 1
        messages.append({"role": "assistant", "content": raw})

        if action.get("is_complete", False):
            metrics["success"] = True
            break

        messages.append({
            "role": "user",
            "content": f"Tool {tool_used} executed. Continue to the next step."
        })
        time.sleep(1)

    return metrics


def run_smart_lite_agent(issue, repo):
    metrics = {
        "task_id": repo,
        "success": False,
        "total_steps": 0,
        "invalid_tool_calls": 0,
        "phase_violations": 0,
        "verification_executed": False,
        "total_tokens": 0,
        "avg_tool_reward": 0.0,
        "trajectory": []
    }

    for phase_name in PHASE_ORDER:
        phase = PHASES[phase_name]
        print(f"  Phase: {phase_name.upper()}")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"{phase['prompt']}\n\nIssue in {repo}:\n{issue}"
            }
        ]

        for step in range(3):
            response = call_openai_with_retry(messages)
            raw = response.choices[0].message.content
            metrics["total_tokens"] += response.usage.total_tokens

            try:
                clean = raw.replace("```json","").replace("```","").strip()
                action = json.loads(clean)
            except:
                metrics["invalid_tool_calls"] += 1
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": "Your response was not valid JSON. Please respond with valid JSON only."
                })
                continue

            tool_used = action.get("tool", "")

            if tool_used in phase["forbidden_tools"]:
                metrics["phase_violations"] += 1
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": f"PHASE VIOLATION: You cannot use {tool_used} in the {phase_name} phase. Use only: {phase['allowed_tools']}"
                })
                continue

            if tool_used not in ALL_TOOLS:
                metrics["invalid_tool_calls"] += 1
            if tool_used == "run_tests":
                metrics["verification_executed"] = True

            metrics["trajectory"].append({
                "step": metrics["total_steps"] + 1,
                "phase": phase_name,
                "thought": action.get("thought", ""),
                "tool": tool_used,
                "args": action.get("args", {})
            })
            metrics["total_steps"] += 1
            messages.append({"role": "assistant", "content": raw})

            if action.get("is_complete", False):
                break

            messages.append({
                "role": "user",
                "content": f"Tool {tool_used} executed. Continue in the {phase_name} phase."
            })
            time.sleep(1)

    metrics["success"] = metrics["verification_executed"]
    return metrics


def run_smart_full_agent(issue, repo):
    memory = EntityMemory()
    metrics = {
        "task_id": repo,
        "success": False,
        "total_steps": 0,
        "invalid_tool_calls": 0,
        "phase_violations": 0,
        "verification_executed": False,
        "total_tokens": 0,
        "avg_tool_reward": 0.0,
        "memory_retrievals": 0,
        "tool_reward_scores": [],
        "trajectory": []
    }

    memory.add_record(issue, source="issue", phase="init")

    for phase_name in PHASE_ORDER:
        phase = PHASES[phase_name]
        print(f"  Phase: {phase_name.upper()}")

        relevant_memories = memory.retrieve(issue, top_k=3)
        metrics["memory_retrievals"] += len(relevant_memories)

        memory_context = ""
        if relevant_memories:
            memory_context = "\n\nRelevant context from memory:\n"
            for mem in relevant_memories:
                memory_context += f"- [{mem['phase']}] {mem['content'][:100]}\n"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"{phase['prompt']}\n\nIssue in {repo}:\n{issue}{memory_context}"
            }
        ]

        for step in range(3):
            response = call_openai_with_retry(messages)
            raw = response.choices[0].message.content
            metrics["total_tokens"] += response.usage.total_tokens

            try:
                clean = raw.replace("```json","").replace("```","").strip()
                action = json.loads(clean)
            except:
                metrics["invalid_tool_calls"] += 1
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": "Your response was not valid JSON. Please respond with valid JSON only."
                })
                continue

            tool_used = action.get("tool", "")
            thought = action.get("thought", "")

            if tool_used in phase["forbidden_tools"]:
                metrics["phase_violations"] += 1
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": f"PHASE VIOLATION: You cannot use {tool_used} in the {phase_name} phase. Use only: {phase['allowed_tools']}"
                })
                continue

            if tool_used not in ALL_TOOLS:
                metrics["invalid_tool_calls"] += 1
            if tool_used == "run_tests":
                metrics["verification_executed"] = True

            tool_reward = 0.0
            if tool_used in ALL_TOOLS:
                tool_reward += 0.25
            if tool_used in phase["allowed_tools"]:
                tool_reward += 0.25
            if tool_used not in phase["forbidden_tools"]:
                tool_reward += 0.25
            if thought:
                tool_reward += 0.25
            metrics["tool_reward_scores"].append(tool_reward)

            memory.add_record(
                content=f"{thought} used {tool_used}",
                source=tool_used,
                phase=phase_name
            )

            metrics["trajectory"].append({
                "step": metrics["total_steps"] + 1,
                "phase": phase_name,
                "thought": thought,
                "tool": tool_used,
                "args": action.get("args", {}),
                "tool_reward": tool_reward
            })
            metrics["total_steps"] += 1
            messages.append({"role": "assistant", "content": raw})

            if action.get("is_complete", False):
                break

            messages.append({
                "role": "user",
                "content": f"Tool {tool_used} executed. Continue in the {phase_name} phase."
            })
            time.sleep(1)

    if metrics["tool_reward_scores"]:
        metrics["avg_tool_reward"] = sum(
            metrics["tool_reward_scores"]) / len(
            metrics["tool_reward_scores"])

    metrics["success"] = metrics["verification_executed"]
    return metrics

print("All three agents defined successfully")

All three agents defined successfully


In [6]:
def run_experiment(dataset, agent_func, num_tasks=30, agent_name="Agent"):
    results = []
    save_file = f"{agent_name.lower().replace('-','_')}_results.csv"

    print(f"\nRunning {agent_name} on {num_tasks} tasks...")
    print(f"Model: {MODEL}")
    print("-" * 50)

    for i in range(num_tasks):
        task = dataset[i]
        repo = task["repo"]
        issue = task["problem_statement"]

        print(f"\nTask {i+1}/{num_tasks}: {repo}")

        try:
            metrics = agent_func(issue, repo)
            metrics["agent_name"] = agent_name
            metrics["task_number"] = i + 1
            results.append(metrics)

            print(f"  Steps: {metrics['total_steps']} | "
                  f"Invalid: {metrics['invalid_tool_calls']} | "
                  f"Violations: {metrics['phase_violations']} | "
                  f"Verified: {metrics['verification_executed']} | "
                  f"Tokens: {metrics['total_tokens']}")

        except Exception as e:
            print(f"  Task failed: {e}")
            results.append({
                "agent_name": agent_name,
                "task_number": i + 1,
                "task_id": repo,
                "success": False,
                "total_steps": 0,
                "invalid_tool_calls": 0,
                "phase_violations": 0,
                "verification_executed": False,
                "total_tokens": 0,
                "avg_tool_reward": 0.0,
                "error": str(e)
            })

        pd.DataFrame(results).to_csv(save_file, index=False)

    print(f"\nDone. Results saved to {save_file}")
    return results

print("Experiment runner defined successfully")

Experiment runner defined successfully


In [7]:
print("Starting full experiment on 30 tasks...")
print("=" * 50)

baseline_results = run_experiment(
    dataset=dataset,
    agent_func=run_baseline_agent,
    num_tasks=30,
    agent_name="Baseline"
)

smart_lite_results = run_experiment(
    dataset=dataset,
    agent_func=run_smart_lite_agent,
    num_tasks=30,
    agent_name="SMART-lite"
)

smart_full_results = run_experiment(
    dataset=dataset,
    agent_func=run_smart_full_agent,
    num_tasks=30,
    agent_name="SMART-full"
)

baseline_df = pd.DataFrame(baseline_results)
smart_lite_df = pd.DataFrame(smart_lite_results)
smart_full_df = pd.DataFrame(smart_full_results)

print("\nAll experiments complete")

Starting full experiment on 30 tasks...

Running Baseline on 30 tasks...
Model: gpt-3.5-turbo
--------------------------------------------------

Task 1/30: astropy/astropy
  Steps: 7 | Invalid: 1 | Violations: 0 | Verified: True | Tokens: 5390

Task 2/30: astropy/astropy
  Steps: 4 | Invalid: 0 | Violations: 0 | Verified: True | Tokens: 3363

Task 3/30: astropy/astropy
  Steps: 4 | Invalid: 0 | Violations: 0 | Verified: True | Tokens: 3135

Task 4/30: astropy/astropy
  Steps: 4 | Invalid: 0 | Violations: 0 | Verified: True | Tokens: 4423

Task 5/30: astropy/astropy
  Steps: 4 | Invalid: 0 | Violations: 0 | Verified: True | Tokens: 1798

Task 6/30: astropy/astropy
  Steps: 6 | Invalid: 1 | Violations: 0 | Verified: True | Tokens: 5808

Task 7/30: django/django
  Steps: 5 | Invalid: 1 | Violations: 0 | Verified: True | Tokens: 2830

Task 8/30: django/django
  Steps: 6 | Invalid: 1 | Violations: 0 | Verified: True | Tokens: 3396

Task 9/30: django/django
  Steps: 5 | Invalid: 0 | Violati

In [8]:
print("\n")
print("=" * 70)
print("FULL EXPERIMENT RESULTS: 30 Tasks")
print("=" * 70)
print(f"{'Metric':<30} {'Baseline':>10} {'SMART-lite':>10} {'SMART-full':>10}")
print("-" * 70)
print(f"{'Avg steps per task':<30} "
      f"{baseline_df['total_steps'].mean():>10.1f} "
      f"{smart_lite_df['total_steps'].mean():>10.1f} "
      f"{smart_full_df['total_steps'].mean():>10.1f}")
print(f"{'Avg invalid tool calls':<30} "
      f"{baseline_df['invalid_tool_calls'].mean():>10.1f} "
      f"{smart_lite_df['invalid_tool_calls'].mean():>10.1f} "
      f"{smart_full_df['invalid_tool_calls'].mean():>10.1f}")
print(f"{'Avg phase violations':<30} "
      f"{baseline_df['phase_violations'].mean():>10.1f} "
      f"{smart_lite_df['phase_violations'].mean():>10.1f} "
      f"{smart_full_df['phase_violations'].mean():>10.1f}")
print(f"{'Verification rate':<30} "
      f"{baseline_df['verification_executed'].mean()*100:>9.1f}% "
      f"{smart_lite_df['verification_executed'].mean()*100:>9.1f}% "
      f"{smart_full_df['verification_executed'].mean()*100:>9.1f}%")
print(f"{'Avg tokens per task':<30} "
      f"{baseline_df['total_tokens'].mean():>10.0f} "
      f"{smart_lite_df['total_tokens'].mean():>10.0f} "
      f"{smart_full_df['total_tokens'].mean():>10.0f}")
print(f"{'Avg tool reward':<30} "
      f"{'N/A':>10} "
      f"{'N/A':>10} "
      f"{smart_full_df['avg_tool_reward'].mean():>10.2f}")
print("=" * 70)



FULL EXPERIMENT RESULTS: 30 Tasks
Metric                           Baseline SMART-lite SMART-full
----------------------------------------------------------------------
Avg steps per task                    4.8        6.4        7.0
Avg invalid tool calls                1.0        0.6        0.9
Avg phase violations                  0.0        0.0        0.0
Verification rate                   90.0%     100.0%     100.0%
Avg tokens per task                  4021       4040       4914
Avg tool reward                       N/A        N/A       0.95


In [9]:
def generate_latex_table(baseline_df, smart_lite_df, smart_full_df):
    latex = r"""
\begin{table}[t]
\centering
\small
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{5pt}
\begin{tabular}{lccccc}
\toprule
\textbf{Configuration} &
\textbf{Avg Steps} &
\textbf{Invalid Calls} &
\textbf{Phase Violations} &
\textbf{Verification Rate} &
\textbf{Avg Tokens} \\
\midrule
"""
    configs = {
        "Baseline": baseline_df,
        "SMART-lite": smart_lite_df,
        "SMART-full": smart_full_df
    }

    for name, df in configs.items():
        latex += (
            f"{name} & "
            f"{df['total_steps'].mean():.1f} & "
            f"{df['invalid_tool_calls'].mean():.1f} & "
            f"{df['phase_violations'].mean():.1f} & "
            f"{df['verification_executed'].mean()*100:.1f}\\% & "
            f"{df['total_tokens'].mean():.0f} \\\\\n"
        )

    latex += r"""
\bottomrule
\end{tabular}
\caption{Ablation results on 30 SWE-bench Lite tasks.
SMART-lite uses scaffolded execution only.
SMART-full adds entity-centric memory and process-level supervision.
Lower is better for Avg Steps, Invalid Calls, Phase Violations,
and Avg Tokens. Higher is better for Verification Rate.}
\label{tab:results}
\end{table}
"""
    return latex


latex_table = generate_latex_table(
    baseline_df,
    smart_lite_df,
    smart_full_df
)

print("LaTeX table for your paper:")
print("=" * 60)
print(latex_table)
print("=" * 60)

with open("results_table.tex", "w") as f:
    f.write(latex_table)
print("Table saved to results_table.tex")

print("\nImprovement of SMART-full over Baseline:")
print("-" * 40)
invalid_imp = ((baseline_df['invalid_tool_calls'].mean() -
                smart_full_df['invalid_tool_calls'].mean()) /
               max(baseline_df['invalid_tool_calls'].mean(), 0.001) * 100)
violation_imp = ((baseline_df['phase_violations'].mean() -
                  smart_full_df['phase_violations'].mean()) /
                 max(baseline_df['phase_violations'].mean(), 0.001) * 100)
verify_imp = (smart_full_df['verification_executed'].mean() -
              baseline_df['verification_executed'].mean()) * 100
token_imp = ((baseline_df['total_tokens'].mean() -
              smart_full_df['total_tokens'].mean()) /
             max(baseline_df['total_tokens'].mean(), 0.001) * 100)

print(f"Invalid tool calls reduced by:  {invalid_imp:.1f}%")
print(f"Phase violations reduced by:    {violation_imp:.1f}%")
print(f"Verification rate improved by:  {verify_imp:.1f}%")
print(f"Token usage difference:         {token_imp:.1f}%")
print("\nCopy the LaTeX table above and paste into your paper as Table 4")

LaTeX table for your paper:

\begin{table}[t]
\centering
\small
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{5pt}
\begin{tabular}{lccccc}
\toprule
\textbf{Configuration} &
\textbf{Avg Steps} &
\textbf{Invalid Calls} &
\textbf{Phase Violations} &
\textbf{Verification Rate} &
\textbf{Avg Tokens} \\
\midrule
Baseline & 4.8 & 1.0 & 0.0 & 90.0\% & 4021 \\
SMART-lite & 6.4 & 0.6 & 0.0 & 100.0\% & 4040 \\
SMART-full & 7.0 & 0.9 & 0.0 & 100.0\% & 4914 \\

\bottomrule
\end{tabular}
\caption{Ablation results on 30 SWE-bench Lite tasks.
SMART-lite uses scaffolded execution only.
SMART-full adds entity-centric memory and process-level supervision.
Lower is better for Avg Steps, Invalid Calls, Phase Violations,
and Avg Tokens. Higher is better for Verification Rate.}
\label{tab:results}
\end{table}

Table saved to results_table.tex

Improvement of SMART-full over Baseline:
----------------------------------------
Invalid tool calls reduced by:  6.7%
Phase violations reduced by:    0.0%
